In [20]:
import ee
import ffmpeg
import geemap
from geemap import cartoee

In [21]:
ee.Authenticate()
ee.Initialize(project="qflfw-511214")

In [27]:
lon_min = -1.23
lon_max = -0.89
lat_min = 44.72
lat_max = 44.96

bbox = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)
# # Create an ee.Geometry.
# polygon = ee.Geometry.Polygon(
#     [[[-35, -10], [35, -10], [35, 10], [-35, 10], [-35, -10]]]
# )

# # Create a Feature from the Geometry.
# poly_feature = ee.Feature(polygon, {'foo': 42, 'bar': 'tart'})

dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2022-07-01', '2022-07-31')
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .filterBounds(bbox)
)

first_img = dataset.first()

dataset.size().getInfo()

4

In [23]:
#Checking the projection is of the desired type:
our_crs="EPSG:32630"
bands = ["B4", "B3", "B2"]
for band in bands:
    if (first_img.select(band).projection().getInfo()["crs"] != our_crs):
        raise Exception("The projection type does not match the desired one!")
    else:
        print(f"{band} OK!")

B4 OK!
B3 OK!
B2 OK!


In [24]:
visualization = {
    'min': 0.0,
    'max': 5000,
    'bands': bands,
}


m = geemap.Map()
m.center_object(bbox, zoom=9)

m.add_time_slider(dataset, vis_params=visualization ,time_interval=3)


In [25]:
display(m)

Map(center=[44.83995872472239, -1.0600000000001695], controls=(WidgetControl(options=['position', 'transparent…

In [28]:
geemap.create_timelapse(collection=dataset,
    bands=bands,
    dimensions=1080,
    region=dataset.geometry(),
    start_date='2022-07-01', 
    end_date='2022-07-31', 
    out_gif="fire.gif",
    loop=0,
    frequency='day',
    frames_per_second=1)

Total number of images: 4

ffmpeg is not installed on your computer.


'/home/ensai/Documents/Quantifying-forest-loss-from-wildfires/fire.gif'